In [1]:

import pandas as pd
import numpy as np
import os

# Load the original raw dataset
df_raw = pd.read_csv(
    "../data/raw/Maternal_Health_Risk.csv"
)

# Work on a copy, never on the original file
df = df_raw.copy()

# Remove accidental spaces around column names
df.columns = df.columns.str.strip()

print("Original dataset shape:", df.shape)
display(df.head())


Original dataset shape: (6103, 11)


,Patient ID,Name,Age,Body Temperature(F),Heart rate(bpm),Systolic Blood Pressure(mm Hg),Diastolic Blood Pressure(mm Hg),BMI(kg/m 2),Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl),Status
0,101,Bhavana,20,97.5,91,161,100,24.9,41,5.8,high risk
1,102,Mithila,45,97.7,99,99,94,22.1,36,5.7,high risk
2,103,Phoolan,29,98.6,84,129,87,19.0,42,6.4,mid risk
3,104,Joyonti,26,99.5,135,161,101,23.7,46,4.5,high risk
4,105,Vana,38,102.5,51,106,91,18.8,38,4.3,high risk


In [2]:

# Identify personal identifier columns
identifier_columns = ["Patient ID", "Name"]

# Remove only those that exist
df = df.drop(
    columns=identifier_columns,
    errors="ignore"
)

print("Shape after removing identifiers:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())


Shape after removing identifiers: (6103, 9)
Remaining columns:
['Age', 'Body Temperature(F)', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']


In [3]:

# Expected numerical columns in your dataset
numeric_columns = [
    "Age",
    "Body Temperature(F)",
    "Heart rate(bpm)",
    "Systolic Blood Pressure(mm Hg)",
    "Diastolic Blood Pressure(mm Hg)",
    "BMI(kg/m 2)",
    "Blood Glucose(HbA1c)",
    "Blood Glucose(Fasting hour-mg/dl)"
]

# Convert existing columns to numeric
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Updated data types:")
print(df.dtypes)


Updated data types:
Age                                    int64
Body Temperature(F)                  float64
Heart rate(bpm)                        int64
Systolic Blood Pressure(mm Hg)         int64
Diastolic Blood Pressure(mm Hg)        int64
BMI(kg/m 2)                          float64
Blood Glucose(HbA1c)                   int64
Blood Glucose(Fasting hour-mg/dl)    float64
Status                                object
dtype: object


In [4]:

print("MISSING VALUES:")
display(df.isnull().sum())

print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

print("\nTARGET MISSING VALUES:")
print(df["Status"].isnull().sum())

print("\nTARGET CATEGORIES:")
print(df["Status"].value_counts(dropna=False))


MISSING VALUES:


Age                                  0
Body Temperature(F)                  0
Heart rate(bpm)                      0
Systolic Blood Pressure(mm Hg)       0
Diastolic Blood Pressure(mm Hg)      0
BMI(kg/m 2)                          0
Blood Glucose(HbA1c)                 0
Blood Glucose(Fasting hour-mg/dl)    0
Status                               0
dtype: int64


DUPLICATE ROWS:
1

TARGET MISSING VALUES:
0

TARGET CATEGORIES:
Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64


In [5]:

print("NUMERICAL SUMMARY:")
display(df.describe())

# Count potential outliers using the IQR method
for col in numeric_columns:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = (
            (df[col] < lower) |
            (df[col] > upper)
        ).sum()

        print(f"{col}: {count} possible outliers")


NUMERICAL SUMMARY:


,Age,Body Temperature(F),Heart rate(bpm),Systolic Blood Pressure(mm Hg),Diastolic Blood Pressure(mm Hg),BMI(kg/m 2),Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl)
count,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000
mean,26.425037,98.665574,86.100770,129.218253,87.257578,21.435581,37.904473,5.504752
std,6.390205,1.590983,22.627587,17.234217,7.793099,2.157060,4.400272,0.905327
min,15.000000,39.600000,45.000000,90.000000,9.000000,14.900000,30.000000,3.500000
25%,22.000000,98.600000,72.000000,120.000000,82.000000,19.600000,34.000000,4.800000
50%,25.000000,98.600000,80.000000,128.000000,87.000000,21.300000,38.000000,5.700000
75%,30.000000,98.800000,91.000000,141.000000,92.000000,23.100000,41.000000,6.000000
max,250.000000,104.000000,150.000000,169.000000,142.000000,27.900000,50.000000,8.900000


Age: 86 possible outliers
Body Temperature(F): 1763 possible outliers
Heart rate(bpm): 742 possible outliers
Systolic Blood Pressure(mm Hg): 0 possible outliers
Diastolic Blood Pressure(mm Hg): 149 possible outliers
BMI(kg/m 2): 0 possible outliers
Blood Glucose(HbA1c): 0 possible outliers
Blood Glucose(Fasting hour-mg/dl): 27 possible outliers


In [6]:

os.makedirs("../data/interim", exist_ok=True)

df.to_csv(
    "../data/interim/maternal_health_initial_clean.csv",
    index=False
)

print("Initial cleaned working dataset saved successfully!")
print("Final shape:", df.shape)


Initial cleaned working dataset saved successfully!
Final shape: (6103, 9)
